In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
# X, y = make_friedman1(n_samples=10, noise=1e-2)
# X, y = make_friedman1(n_samples=100, noise=1e-2)
# X, y = make_friedman1(n_samples=1000, noise=1e-2)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.Newton(
    model=model,
    mu=1e-3,
    lr_init=1e-3,
    damping=True,
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.15452894568443298
epoch:  1, loss: 0.15197725594043732
epoch:  2, loss: 0.1496048867702484
epoch:  3, loss: 0.14717791974544525
epoch:  4, loss: 0.1447756141424179
epoch:  5, loss: 0.14240676164627075
epoch:  6, loss: 0.14010049402713776
epoch:  7, loss: 0.13775500655174255
epoch:  8, loss: 0.13542556762695312
epoch:  9, loss: 0.13313336670398712
epoch:  10, loss: 0.13085408508777618
epoch:  11, loss: 0.12865163385868073
epoch:  12, loss: 0.1264926940202713
epoch:  13, loss: 0.124376080930233
epoch:  14, loss: 0.1223071813583374
epoch:  15, loss: 0.12028712779283524
epoch:  16, loss: 0.11824506521224976
epoch:  17, loss: 0.11625678837299347
epoch:  18, loss: 0.11430708318948746
epoch:  19, loss: 0.11240147054195404
epoch:  20, loss: 0.11053194105625153
epoch:  21, loss: 0.10870150476694107
epoch:  22, loss: 0.10690290480852127
epoch:  23, loss: 0.10513899475336075
epoch:  24, loss: 0.10340392589569092
epoch:  25, loss: 0.10170217603445053
epoch:  26, loss: 0.10003025

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.951971709728241
Test metrics:  R2 = 0.9011400938034058


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(
    model=model,
    mu=1e-3,
    damping=True,
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.05840878188610077
epoch:  1, loss: 0.05129795894026756
epoch:  2, loss: 0.045478228479623795
epoch:  3, loss: 0.03940531983971596
epoch:  4, loss: 0.034984298050403595
epoch:  5, loss: 0.031174391508102417
epoch:  6, loss: 0.017919475212693214
epoch:  7, loss: 0.016252627596259117
epoch:  8, loss: 0.015923235565423965
epoch:  9, loss: 0.006531450431793928
epoch:  10, loss: 0.0052143787033855915
epoch:  11, loss: 0.0041690850630402565
epoch:  12, loss: 0.0034523229114711285
epoch:  13, loss: 0.0029927357099950314
epoch:  14, loss: 0.0026422804221510887
epoch:  15, loss: 0.0023846407420933247
epoch:  16, loss: 0.0021911405492573977
epoch:  17, loss: 0.0020455399062484503
epoch:  18, loss: 0.0019378077704459429
epoch:  19, loss: 0.0018257995834574103
epoch:  20, loss: 0.0017399777425453067
epoch:  21, loss: 0.0016678348183631897
epoch:  22, loss: 0.001596895162947476
epoch:  23, loss: 0.0015351037727668881
epoch:  24, loss: 0.0014867502031847835
epoch:  25, loss: 0.0014

In [8]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9873690009117126
Test metrics:  R2 = 0.8275281190872192
